# Architecture Ablation (AMI)

Reproduces the architecture-ablation table in Section 4.3 of the manuscript. Each variant changes one knob from the default hierarchical-transformer config; all variants share the same loss config (the best one from the loss ablation: `λ_wer=1.0, λ_hard=0.0, λ_soft=0.5, τ=0.1`).

The full sweep runs **8 variants** end-to-end; expect ~3–4 hours on L4 with 50 epochs each. To do a quick smoke test first, run a single variant with `--max-epochs 2` (see below).


## 1. Initial setup (clone + uv env + secrets)

GPU recommendation:
- **L4** (default): best price/performance for these notebooks. ~1.5–3× faster than T4 with no per-second penalty.
- **A100**: ~2× faster than L4, ~2× the price. Use only if you want the full 50-epoch run in under 30 minutes.
- **T4**: works but slow; halve `batch_size` if you OOM.


In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

from google.colab import drive
drive.mount('/content/drive')


## 2. Bring in the AMI interim features from Drive

Reuses the per-expert interim parquets you have already generated and saved to Drive (no re-extraction). Then runs the patched preprocess script that propagates the `_transcription` columns alongside the embeddings/WERs.

> **Adjust `DRIVE_INTERIM_AMI` below** to your actual Drive folder path before running.


In [ ]:
# Path on Drive that contains the three per-expert interim parquets:
#   openai_whisper-base.parquet
#   facebook_hubert-large-ls960-ft.parquet
#   facebook_wav2vec2-base-960h.parquet
DRIVE_INTERIM_AMI = '/content/drive/MyDrive/s2t-tr-dev/data/interim/edinburghcstr_ami'

!mkdir -p data/interim/edinburghcstr_ami
!cp -v "$DRIVE_INTERIM_AMI"/*.parquet data/interim/edinburghcstr_ami/
!ls -lh data/interim/edinburghcstr_ami/


In [ ]:
!uv run python -m src.data.preprocess \
    --dataset edinburghcstr/ami \
    --output_name combined_features_with_transcripts.parquet


## 3. Smoke test (single variant, 2 epochs)

Confirms the full pipeline before committing to the 8-variant sweep.


In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/edinburghcstr_ami/combined_features_with_transcripts.parquet \
    --arch hierarchical_transformer \
    --batch-size 8 --num-workers 2 \
    --max-epochs 2 --limit-batches 16 \
    --experiment-name smoke_test \
    --log-dir logs/architecture_ablation


## 4. Run the architecture ablation

8 trained variants × ~25 min each on L4 ≈ 3–4 h. The orchestrator writes one combined `ablation_results.json` after every variant, so the run is checkpoint-safe — restart and existing variants will retrain (no resume logic; the dataset and loss are identical so this is fine).

> **A100 settings**: `--max-epochs 30` (converges faster), `batch_size 16` in the JSON.
> **T4 settings**: keep epochs at 50 but set `batch_size 4` to avoid OOM.


In [ ]:
!uv run python -m src.experiments.run_ablation \
    --config configs/architecture_ablation.json \
    --output-dir reports/ablations/architecture_ami


## 5. Render the table

Prints one row per variant (test selected_wer, oracle_wer, gap, selection_accuracy). This is the data backing the ablation table in the manuscript.


In [ ]:
import json
from pathlib import Path

results = json.loads(Path('reports/ablations/architecture_ami/ablation_results.json').read_text())
rows = []
for name, entry in results['variants'].items():
    t = entry['test']
    rows.append((name, t['selected_wer'], t['oracle_wer'], t['wer_gap'], t['selection_accuracy']))

print(f"{'variant':<22} {'sel_wer':>8} {'oracle':>8} {'gap':>8} {'sel_acc':>8}")
print('-' * 64)
for r in rows:
    print(f"{r[0]:<22} {r[1]:>8.4f} {r[2]:>8.4f} {r[3]:>8.4f} {r[4]:>8.4f}")


## 6. Backup logs to Drive


In [ ]:
!cp -r reports/ablations/architecture_ami /content/drive/MyDrive/s2t-tr-dev/ablations/architecture_ami_$(date +%Y%m%d_%H%M)
